# 24. End-to-End Case Study: Industrial Sensor Telemetry Anomaly & Failure Prediction

A complete time-series and predictive maintenance pipeline: entity normalization, sensor drift, rolling volatility, and early degradation warnings.


## 1. Objective
Predict impending machine failure (`failure`) from multi-variate sensor telemetry across 10 industrial machines.
Key challenges:
1. Machine-specific baselines (M01 runs hotter than M04).
2. Sensor degradation drift prior to breakdown.
3. Multi-sensor correlation (Vibration spikes + Pressure drop).
4. Strictly chronological evaluation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

df = pd.read_csv('../datasets/machine_sensors/industrial_sensors.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['machine_id', 'timestamp']).reset_index(drop=True)
print(f"Sensor Dataset: {df.shape[0]:,} rows across {df['machine_id'].nunique()} machines | Failures: {df['failure'].sum()}")


## 2. Entity-Level Feature Engineering & Rolling Volatility


In [ ]:
# 1. Grouped Rolling Means & Volatility (Shifted to prevent leakage)
for s in ['temperature', 'vibration', 'pressure']:
    grp = df.groupby('machine_id')[s]
    df[f'{s}_roll_6h_mean'] = grp.transform(lambda x: x.shift(1).rolling(6).mean())
    df[f'{s}_roll_24h_mean'] = grp.transform(lambda x: x.shift(1).rolling(24).mean())
    df[f'{s}_roll_6h_std'] = grp.transform(lambda x: x.shift(1).rolling(6).std())
    # Short-term vs long-term thermal/vibration drift
    df[f'{s}_drift_ratio'] = df[f'{s}_roll_6h_mean'] / (df[f'{s}_roll_24h_mean'] + 1e-5)

# 2. Multi-Sensor Stress Interaction (Vibration / Pressure ratio)
df['vib_press_stress'] = df['vibration_roll_6h_mean'] / (df['pressure_roll_6h_mean'] + 1e-5)

# Drop warm-up window
df_clean = df.dropna().copy()
print(f"Cleaned feature matrix: {df_clean.shape}")


## 3. Chronological Model Evaluation


In [ ]:
unique_times = np.sort(df_clean['timestamp'].unique())
split_time = unique_times[int(len(unique_times) * 0.80)]

train_df = df_clean[df_clean['timestamp'] < split_time]
test_df = df_clean[df_clean['timestamp'] >= split_time]

feat_cols = [c for c in df_clean.columns if c not in ['timestamp', 'machine_id', 'failure']]

clf = HistGradientBoostingClassifier(class_weight='balanced', random_state=42)
clf.fit(train_df[feat_cols], train_df['failure'])

test_probs = clf.predict_proba(test_df[feat_cols])[:, 1]
print("=" * 60)
print(f"SENSOR PREDICTIVE MAINTENANCE TEST ROC-AUC: {roc_auc_score(test_df['failure'], test_probs):.4f}")
print(f"SENSOR PREDICTIVE MAINTENANCE TEST PR-AUC:  {average_precision_score(test_df['failure'], test_probs):.4f}")
print("=" * 60)
